# Create Linear Regression model with Pytorch components

## Operation steps

1. Data gathering
2. Data preprocessing
3. Feature engineering
4. Model training
5. Testing

**Dataset**: [insurance.csv](https://www.kaggle.com/datasets/mirichoi0218/insurance)

In [5]:
# Install kaggle
# !pip install --upgrade pip
# !pip install kaggle kagglehub

In [54]:
# Import libraries

import kagglehub
import os
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split as TTS
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [7]:
# Download latest version
path = kagglehub.dataset_download("mirichoi0218/insurance")
print(f"Path of dataset files: {path}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16.0k/16.0k [00:00<00:00, 52.9kB/s]

Extracting files...
Path of dataset files: /root/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1


In [8]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mirichoi0218/insurance")

print(f"Path of dataset files : {path}")

Path of dataset files : /root/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1


In [9]:
# Current working directory
print(os.getcwd())

/mnt/sdcard/Study/Computer-Vision-with-CNN-TensorFlow-and-PyTorch/06-PyTorch


In [10]:
os.listdir(path)

['insurance.csv']

In [11]:
df = pd.read_csv(os.path.join(path, 'insurance.csv'))
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
df.info()

# NOTE: 'charges' is our target column for prediction

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [13]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [14]:
# Split data frames before encoding to avoid data leakage

train_df, test_df = TTS(df, test_size=0.2, random_state=42)
train_df.shape, test_df.shape

((1070, 7), (268, 7))

In [15]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1070 entries, 560 to 1126
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1070 non-null   int64  
 1   sex       1070 non-null   object 
 2   bmi       1070 non-null   float64
 3   children  1070 non-null   int64  
 4   smoker    1070 non-null   object 
 5   region    1070 non-null   object 
 6   charges   1070 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 66.9+ KB


In [24]:
# Encode categorical variables
cat_cols = list(train_df.select_dtypes(include=['object']).columns)

label_encoder = {}

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoder[col] = le

print(label_encoder)

{'sex': LabelEncoder(), 'smoker': LabelEncoder(), 'region': LabelEncoder()}


In [25]:
train_df[cat_cols].apply(lambda x: x.value_counts())

,sex,smoker,region
0,522.0,850.0,267
1,548.0,220.0,256
2,NaN,NaN,283
3,NaN,NaN,264


In [26]:
# Split features and target 
X_train = train_df.drop(columns=['charges'])
y_train = train_df['charges']
X_test = test_df.drop(columns=['charges'])
y_test = test_df['charges']

In [27]:
print(X_train.head())
print(y_train.head())

      age  sex    bmi  children  smoker  region
560    46    0  19.95         2       0       1
1285   47    0  24.32         0       0       0
1142   52    0  24.86         0       0       2
969    39    0  34.32         5       0       2
486    54    0  21.47         3       0       1
560      9193.83850
1285     8534.67180
1142    27117.99378
969      8596.82780
486     12475.35130
Name: charges, dtype: float64


In [28]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
print(X_train)

[[ 0.47222651 -1.0246016  -1.75652513  0.73433626 -0.50874702 -0.45611589]
 [ 0.54331294 -1.0246016  -1.03308239 -0.91119211 -0.50874702 -1.35325561]
 [ 0.8987451  -1.0246016  -0.94368672 -0.91119211 -0.50874702  0.44102382]
 ...
 [ 1.3252637   0.97598911 -0.89153925 -0.91119211 -0.50874702 -1.35325561]
 [-0.16755139 -1.0246016   2.82086429  0.73433626  1.96561348  1.33816354]
 [ 1.1120044   0.97598911 -0.10932713 -0.91119211 -0.50874702  1.33816354]]


In [33]:
type(X_train), type(y_train), type(y_train.values)

(numpy.ndarray, pandas.core.series.Series, numpy.ndarray)

In [46]:
# Convert numpy array to tensors

# Independent features
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

# Dependent features
# It should be y_train.values or y_test.values to determine shape
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [47]:
print(y_train_tensor.shape)
print(y_test_tensor.shape)

torch.Size([1070, 1])
torch.Size([268, 1])


In [48]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)

torch.Size([1070, 6])
torch.Size([268, 6])


**Create neural network model**

In [56]:
# Define a regression model class
class SimpleNNRegressionModel(nn.Module): 
    def __init__(self, input_dim): # initialize the model; input_dim is the number of input features
        super(SimpleNNRegressionModel, self).__init__() # initialize the parent nn.Module class
        self.network = nn.Sequential(                   # create a sequence of layers that run one after another
                            nn.Linear(input_dim, 64),   # fully connected layer: input_dim features -> 64 neurons
                            nn.ReLU(),                  # apply ReLU activation: negative values become 0
                            nn.Linear(64, 128),         # fully connected layer: 64 neurons -> 128 neurons
                            nn.ReLU(),                  # apply ReLU activation again
                            nn.Linear(128, 1))          # output layer: 128 neurons -> 1 predicted value

    # Define the forward pass; x is the input batch
    def forward(self, x):
        return self.network(x) # pass x through all layers and return the prediction

In [50]:
X_train_tensor.shape

torch.Size([1070, 6])

In [57]:
input_dim = X_train_tensor.shape[1]
model = SimpleNNRegressionModel(input_dim)

In [58]:
print(model)

SimpleNNRegressionModel(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [59]:
# Algorithms for LOSS and OPTIMIZER

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

**Model training**

In [ ]:
# Training loop: repeatedly train the model using the training data
epochs = 30000 # number of times the model will go through the training loop

# Repeat the training process 30,000 times
for epoch in range(epochs):
    model.train() # put the model in training mode
    # NOTE: This matters for layers such as Dropout and BatchNorm

    # Clear gradients left over from the previous training step
    optimizer.zero_grad() # PyTorch accumulates gradients, so we reset them each time

    # Feed training inputs into the model
    predictions = model(X_train_tensor) # the model produces its predicted values

    # Compare predictions with the correct answers
    loss = criterion(predictions, y_train_tensor)  # the loss measures how wrong the predictions are

    # Calculate how each model parameter contributed to the error
    loss.backward() # this computes gradients using backpropagation

    # Update the model's weights using the calculated gradients
    optimizer.step()  # the optimizer adjusts the weights to try to reduce the loss

    # Run this block after every 100 epochs
    if (epoch + 1) % 100 == 0:  
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")  
        # Print the current epoch and loss
        # .item() converts the loss tensor into a regular Python number
        # :.4f displays the loss with 4 digits after the decimal point

Epoch [100/30000], Loss: 43400756.0000
Epoch [200/30000], Loss: 31623990.0000
Epoch [300/30000], Loss: 28788208.0000
Epoch [400/30000], Loss: 27010540.0000
Epoch [500/30000], Loss: 25988252.0000
Epoch [600/30000], Loss: 25313798.0000
Epoch [700/30000], Loss: 24809514.0000
Epoch [800/30000], Loss: 24291518.0000
Epoch [900/30000], Loss: 23808492.0000
Epoch [1000/30000], Loss: 23443866.0000
Epoch [1100/30000], Loss: 23159790.0000
Epoch [1200/30000], Loss: 22895194.0000
Epoch [1300/30000], Loss: 22658354.0000
Epoch [1400/30000], Loss: 22458020.0000
Epoch [1500/30000], Loss: 22166708.0000
Epoch [1600/30000], Loss: 21866256.0000
Epoch [1700/30000], Loss: 21622584.0000
Epoch [1800/30000], Loss: 21418404.0000
Epoch [1900/30000], Loss: 21210336.0000
Epoch [2000/30000], Loss: 21002574.0000
Epoch [2100/30000], Loss: 20803118.0000
Epoch [2200/30000], Loss: 20568150.0000
Epoch [2300/30000], Loss: 20273574.0000
Epoch [2400/30000], Loss: 19997816.0000
Epoch [2500/30000], Loss: 19705174.0000
Epoch [26

In [32]:
# Training loop
epochs = 30000

for epoch in range(epochs):
  model.train()
  optimiser.zero_grad()
  predictions = model(X_train_tensor)
  loss = criterion(predictions, y_train_tensor)
  loss.backward()

  optimiser.step()

  if (epoch+1) % 100 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss : {loss.item():.4f}")

Epoch [100/30000], Loss : 5036951.0000
Epoch [200/30000], Loss : 4972289.0000
Epoch [300/30000], Loss : 4927459.5000
Epoch [400/30000], Loss : 4890543.0000
Epoch [500/30000], Loss : 4856821.0000
Epoch [600/30000], Loss : 4810405.5000
Epoch [700/30000], Loss : 4764390.5000
Epoch [800/30000], Loss : 4717932.5000
Epoch [900/30000], Loss : 4684386.5000
Epoch [1000/30000], Loss : 4649535.0000
Epoch [1100/30000], Loss : 4621969.5000
Epoch [1200/30000], Loss : 4580249.0000
Epoch [1300/30000], Loss : 4545992.0000
Epoch [1400/30000], Loss : 4523264.5000
Epoch [1500/30000], Loss : 4484869.0000
Epoch [1600/30000], Loss : 4445893.0000
Epoch [1700/30000], Loss : 4413395.5000
Epoch [1800/30000], Loss : 4407777.0000
Epoch [1900/30000], Loss : 4349853.5000
Epoch [2000/30000], Loss : 4324418.0000
Epoch [2100/30000], Loss : 4303249.0000
Epoch [2200/30000], Loss : 4269742.0000
Epoch [2300/30000], Loss : 4241508.0000
Epoch [2400/30000], Loss : 4233202.0000
Epoch [2500/30000], Loss : 4191558.7500
Epoch [26

In [35]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_test_numpy = y_test_tensor.numpy()

# Calculate metrics
mse = mean_squared_error(y_test_numpy, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test_numpy, y_pred)
r2 = r2_score(y_test_numpy, y_pred)

print(f"MSE : {mse}")
print(f"RMSE : {rmse}")
print(f"MAE : {mae}")
print(f"R2-Score : {r2}")

# 0 --> 0

MSE : 82683528.0
RMSE : 9093.048333754748
MAE : 6030.34326171875
R2-Score : 0.467412531375885


In [36]:
def predict_charges(age, sex, bmi, children, smoker,region):
  input_data = pd.DataFrame([[age, sex, bmi, children, smoker,region]],
               columns=['age', 'sex', 'bmi', 'children', 'smoker', 'region'])

  for col in ['sex', 'smoker', 'region']:
    input_data[col] = label_encoder[col].transform(input_data[col])
  input_data = scaler.transform(input_data)
  input_tensor = torch.tensor(input_data, dtype=torch.float32)
  predicted_charge = model(input_tensor).item()
  return predicted_charge


In [48]:
predicted = predict_charges(50, 'female', 27.9, 0, 'yes', 'southwest')

In [49]:
print(f"Predicted insurance charge: ${predicted:.2f}")

Predicted insurance charge: $35789.00
